# S'mile — fluctuating travel price models

Two models that predict a **price range** (p10 / p50 / p90) for transport whose price moves around:

| Model | Training data (public) | Target |
|---|---|---|
| Rideshare | NYC TLC High-Volume For-Hire trip records (Uber/Lyft), two months sampled | base passenger fare (before tax, tolls, fees, tip) |
| Airfare | U.S. DOT Consumer Airfare Report, Table 1a (all U.S. airport-pair markets) | average fare for the airport pair and quarter |

Method follows the UrbanLens split: **the model produces evidence, plain code decides.** The app shows the predicted range as an estimate; it is never the exact amount a traveler approves.

Validation is temporal (train on the past, test on the most recent period), with a simple rate-card baseline to beat. Trees are exported to a compact JSON format and re-evaluated in pure Python here to prove the TypeScript evaluator has the same numbers to match.

In [1]:
import os, sys, json, time, math, urllib.request, warnings
# This server has system copies of numexpr/bottleneck built for NumPy 1.x; pandas only uses them
# as optional speedups, so mark them unavailable instead of letting them print import errors.
sys.modules["numexpr"] = None; sys.modules["bottleneck"] = None
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

ROOT = os.path.expanduser("~/smile-price-models"); DATA = f"{ROOT}/data"; OUT = f"{ROOT}/models"
os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
SEED = 7
DEVICE = "cuda" if xgb.build_info().get("USE_CUDA") else "cpu"
QUANTILES = (0.1, 0.5, 0.9)
print("xgboost", xgb.__version__, "| device", DEVICE, "| pandas", pd.__version__, "| numpy", np.__version__)

xgboost 3.2.0 | device cuda | pandas 2.3.3 | numpy 2.4.2


In [2]:
def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    tmp = path + ".part"
    with urllib.request.urlopen(url, timeout=300) as r, open(tmp, "wb") as f:
        while True:
            b = r.read(1 << 22)
            if not b:
                break
            f.write(b)
    os.replace(tmp, path)
    return path

def train_quantiles(Xtr, ytr, Xva, yva, params):
    models = {}
    for q in QUANTILES:
        m = xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, tree_method="hist", device=DEVICE, random_state=SEED, **params)
        m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
        models[q] = m
    return models

def compact(model):
    """XGBoost trees -> compact JSON the app evaluates in TypeScript (no Python at runtime)."""
    j = json.loads(model.get_booster().save_raw("json").decode())
    learner = j["learner"]
    base = float(str(learner["learner_model_param"]["base_score"]).strip("[]").split(",")[0])
    trees = []
    for t in learner["gradient_booster"]["model"]["trees"]:
        trees.append({"l": t["left_children"], "r": t["right_children"], "f": t["split_indices"],
                      "c": [float(x) for x in t["split_conditions"]], "d": [int(x) for x in t["default_left"]]})
    return {"base": base, "trees": trees}

def predict_compact(cm, X):
    """Reference evaluator: go left when x < threshold (float32, as XGBoost does); missing follows default_left."""
    X = np.asarray(X, dtype=np.float32); out = np.full(len(X), cm["base"], dtype=np.float64)
    for t in cm["trees"]:
        L, R, F, C, D = t["l"], t["r"], t["f"], np.asarray(t["c"], dtype=np.float32), t["d"]
        for i, row in enumerate(X):
            n = 0
            while L[n] != -1:
                v = row[F[n]]
                n = (L[n] if D[n] else R[n]) if np.isnan(v) else (L[n] if v < C[n] else R[n])
            out[i] += float(C[n])
    return out

def interval_report(name, y, p10, p50, p90, baseline=None):
    r = {"n_test": int(len(y)), "mae_p50": float(mean_absolute_error(y, p50)), "r2_p50": float(r2_score(y, p50)),
         "mape_p50": float(np.mean(np.abs(p50 - y) / np.maximum(y, 1))),
         "coverage_p10_p90": float(np.mean((y >= p10) & (y <= p90))), "median_band_width": float(np.median(p90 - p10))}
    if baseline is not None:
        r["mae_baseline"] = float(mean_absolute_error(y, baseline)); r["r2_baseline"] = float(r2_score(y, baseline))
    print(name, json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in r.items()}))
    return r

## 1 · Rideshare fares (NYC TLC high-volume for-hire trips)

Real Uber and Lyft trips with the fare the rider was charged, including surge. Features are things S'mile knows before the trip: distance and drive time (from the routing engine), request hour and weekday, month, whether an airport is involved, shared-ride request, and the company.

In [3]:
def tlc_url(ym): return f"https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_{ym}.parquet"
def exists(url):
    try:
        with urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=30) as r:
            return r.status == 200
    except Exception:
        return False

cands = [f"{y}-{m:02d}" for y in (2026, 2025, 2024) for m in range(12, 0, -1)]
latest = next(ym for ym in cands if exists(tlc_url(ym)))
y, m = int(latest[:4]), int(latest[5:])
earlier = f"{y if m > 6 else y - 1}-{(m - 6) if m > 6 else (m + 6):02d}"
MONTHS = [earlier, latest]
print("training months:", MONTHS)
for ym in MONTHS:
    p = download(tlc_url(ym), f"{DATA}/fhvhv_{ym}.parquet"); print(ym, round(os.path.getsize(p) / 2**20), "MB")

training months: ['2026-02', '2026-08']
2026-02 455 MB
2026-08 479 MB


In [4]:
AIRPORT_ZONES = {1, 132, 138}   # EWR, JFK, LGA taxi zones
cols = ["hvfhs_license_num", "request_datetime", "PULocationID", "DOLocationID", "trip_miles", "trip_time",
        "base_passenger_fare", "shared_request_flag"]
frames = []
for ym in MONTHS:
    df = pd.read_parquet(f"{DATA}/fhvhv_{ym}.parquet", columns=cols)
    df = df[df.hvfhs_license_num.isin(["HV0003", "HV0005"])]          # Uber, Lyft
    df = df.sample(n=min(len(df), 3_000_000), random_state=SEED)
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)
print("rows sampled:", len(raw))

d = pd.DataFrame({
    "miles": raw.trip_miles.astype("float32"),
    "minutes": (raw.trip_time / 60).astype("float32"),
    "hour": raw.request_datetime.dt.hour.astype("float32"),
    "dow": raw.request_datetime.dt.dayofweek.astype("float32"),
    "month": raw.request_datetime.dt.month.astype("float32"),
    "airport": (raw.PULocationID.isin(AIRPORT_ZONES) | raw.DOLocationID.isin(AIRPORT_ZONES)).astype("float32"),
    "shared": (raw.shared_request_flag == "Y").astype("float32"),
    "lyft": (raw.hvfhs_license_num == "HV0005").astype("float32"),
    "fare": raw.base_passenger_fare.astype("float32"),
    "ts": raw.request_datetime,
})
ok = d.miles.between(0.3, 100) & d.minutes.between(2, 240) & d.fare.between(2.5, 500) & ((d.fare / d.miles) < 30)
d = d[ok].reset_index(drop=True)
print("rows after cleaning:", len(d), "| dropped", int((~ok).sum()))
d.describe().T[["mean", "50%", "min", "max"]]

rows sampled: 6000000


rows after cleaning: 5938855 | dropped 61145


,mean,50%,min,max
miles,4.850729,2.85,0.3,99.980003
minutes,19.466936,15.866667,2.0,237.449997
hour,13.327835,14.0,0.0,23.0
dow,3.2281,3.0,0.0,6.0
month,5.004477,8.0,1.0,9.0
airport,0.080607,0.0,0.0,1.0
shared,0.016504,0.0,0.0,1.0
lyft,0.284,0.0,0.0,1.0
fare,26.746063,19.280001,2.5,498.019989
ts,2026-05-17 06:21:38.228823,2026-08-01 00:44:27,2026-01-31 23:09:00,2026-09-01 00:00:00


In [5]:
RS_FEATURES = ["miles", "minutes", "hour", "dow", "month", "airport", "shared", "lyft"]
cut = d.ts.max() - pd.Timedelta(days=7)          # temporal split: the most recent week is the test set
tr, te = d[d.ts < cut], d[d.ts >= cut]
va = tr.sample(frac=0.05, random_state=SEED); tr = tr.drop(va.index)
print("train", len(tr), "| valid", len(va), "| test", len(te), "| test starts", cut)

base = LinearRegression().fit(tr[["miles", "minutes"]], tr.fare)   # a plain per-mile + per-minute rate card
print("rate card: $%.2f + $%.2f/mi + $%.2f/min" % (base.intercept_, *base.coef_))

t0 = time.time()
rs = train_quantiles(tr[RS_FEATURES], tr.fare, va[RS_FEATURES], va.fare,
                     dict(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8, max_bin=256))
print("trained in %.1fs on %s" % (time.time() - t0, DEVICE))
P = {q: rs[q].predict(te[RS_FEATURES]) for q in QUANTILES}
rs_metrics = interval_report("rideshare", te.fare.values, P[0.1], P[0.5], P[0.9], base.predict(te[["miles", "minutes"]]))

train 5009476 | valid 263657 | test 665722 | test starts 2026-08-25 00:00:00


rate card: $-1.65 + $0.46/mi + $1.34/min


trained in 11.5s on cuda


rideshare {"n_test": 665722, "mae_p50": 5.5355, "r2_p50": 0.8257, "mape_p50": 0.2096, "coverage_p10_p90": 0.7882, "median_band_width": 12.4031, "mae_baseline": 7.1573, "r2_baseline": 0.7438}


In [6]:
# Price swings by hour: the median fare for a fixed 8-mile / 25-minute ride, all week.
probe = pd.DataFrame([{"miles": 8, "minutes": 25, "hour": h, "dow": dw, "month": 6, "airport": 0, "shared": 0, "lyft": 0}
                      for dw in range(7) for h in range(24)], dtype="float32")
probe["p50"] = rs[0.5].predict(probe[RS_FEATURES])
print(probe.pivot(index="hour", columns="dow", values="p50").round(2).rename(columns=dict(enumerate("Mon Tue Wed Thu Fri Sat Sun".split()))).to_string())
imp = pd.Series(rs[0.5].get_booster().get_score(importance_type="gain")).sort_values(ascending=False)
print("\nfeature importance (gain):"); print(imp.round(1).to_string())

dow         Mon        Tue        Wed        Thu        Fri        Sat        Sun
hour                                                                             
0.0   30.290001  30.620001  30.969999  31.000000  31.799999  32.250000  33.630001
1.0   30.350000  30.639999  30.990000  31.040001  31.850000  32.299999  33.459999
2.0   30.370001  30.629999  30.980000  31.030001  31.860001  32.380001  33.509998
3.0   30.469999  30.660000  30.969999  31.020000  31.830000  32.750000  33.980000
4.0   32.230000  32.290001  32.320000  32.360001  31.820000  32.740002  34.270000
5.0   32.160000  32.240002  32.110001  32.139999  31.510000  31.570000  32.419998
6.0   32.520000  32.689999  32.380001  32.349998  31.200001  30.799999  31.600000
7.0   32.790001  33.000000  32.740002  32.590000  31.340000  30.600000  30.610001
8.0   32.459999  32.580002  32.439999  32.360001  31.100000  30.639999  30.360001
9.0   32.270000  32.470001  32.349998  32.189999  31.139999  30.790001  30.670000
10.0  32.340000 

## 2 · Airfares (U.S. DOT Consumer Airfare Report, Table 1a)

Quarterly average fares for every U.S. airport-pair market. Airport-level features (traffic and fare per mile) are computed **on training years only** and exported with the model so the app can price any pair of airports, including pairs the table never saw.

In [7]:
AF_URL = "https://data.transportation.gov/resource/tfrh-tu9e.csv?$limit=2000000"
af_path = download(AF_URL, f"{DATA}/airfare_table1a.csv")
af = pd.read_csv(af_path, low_memory=False)
print(af.shape); print(list(af.columns))

(262087, 31)
['tbl', 'year', 'quarter', 'citymarketid_1', 'citymarketid_2', 'city1', 'city2', 'airportid_1', 'airportid_2', 'airport_1', 'airport_2', 'nsmiles', 'passengers', 'fare', 'carrier_lg', 'large_ms', 'fare_lg', 'carrier_low', 'lf_ms', 'fare_low', 'location_1', 'location_1_address', 'location_1_city', 'location_1_state', 'location_1_zip', 'location_2', 'location_2_address', 'location_2_city', 'location_2_state', 'location_2_zip', 'tbl1apk']


In [8]:
af = af.rename(columns=str.lower)
need = ["year", "quarter", "airport_1", "airport_2", "city1", "city2", "nsmiles", "passengers", "fare"]
af = af[need].dropna()
af = af[(af.year >= 2015) & af.fare.between(20, 2000) & (af.nsmiles > 50)].copy()
af["t"] = af.year + (af.quarter - 1) / 4
test_from = af.t.max() - 0.75                                   # temporal split: the last 4 quarters are the test set
af_tr, af_te = af[af.t < test_from], af[af.t >= test_from]

# Airport coordinates: OurAirports (public domain), matched on IATA code.
oa = pd.read_csv(download("https://davidmegginson.github.io/ourairports-data/airports.csv", f"{DATA}/ourairports.csv"),
                 usecols=["iata_code", "latitude_deg", "longitude_deg", "type"], low_memory=False)
oa = oa.dropna(subset=["iata_code"]).sort_values("type").drop_duplicates("iata_code").set_index("iata_code")

# airport stats from training years only (no leakage), both directions of every pair
both = pd.concat([af_tr.rename(columns={"airport_1": "ap", "city1": "city"}), af_tr.rename(columns={"airport_2": "ap", "city2": "city"})])
st = both.groupby("ap").agg(pax=("passengers", "sum"), fare=("fare", "sum"), miles=("nsmiles", "sum"), city=("city", "first"))
st["fpm"] = st.fare / st.miles
st["log_pax"] = np.log1p(st.pax)
st = st.join(oa[["latitude_deg", "longitude_deg"]].rename(columns={"latitude_deg": "lat", "longitude_deg": "lon"}))
med = {"log_pax": float(st.log_pax.median()), "fpm": float(st.fpm.median())}
print("airports:", len(st), "| with coordinates:", int(st.lat.notna().sum()))

def af_features(df):
    o = st.reindex(df.airport_1); dd = st.reindex(df.airport_2)
    return pd.DataFrame({
        "nsmiles": df.nsmiles.values, "quarter": df.quarter.values, "year": df.year.values,
        "o_log_pax": o.log_pax.fillna(med["log_pax"]).values, "d_log_pax": dd.log_pax.fillna(med["log_pax"]).values,
        "o_fpm": o.fpm.fillna(med["fpm"]).values, "d_fpm": dd.fpm.fillna(med["fpm"]).values,
    }, dtype="float32")

AF_FEATURES = ["nsmiles", "quarter", "year", "o_log_pax", "d_log_pax", "o_fpm", "d_fpm"]
Xtr, Xte = af_features(af_tr), af_features(af_te)
va_mask = np.random.default_rng(SEED).random(len(Xtr)) < 0.1
afm = train_quantiles(Xtr[~va_mask], af_tr.fare.values[~va_mask], Xtr[va_mask], af_tr.fare.values[va_mask],
                      dict(n_estimators=300, max_depth=6, learning_rate=0.06, subsample=0.8))
af_base = LinearRegression().fit(af_tr[["nsmiles"]], af_tr.fare)
AP = {q: afm[q].predict(Xte) for q in QUANTILES}
print("train", len(af_tr), "| test", len(af_te), "| test from", round(test_from, 2))
af_metrics = interval_report("airfare", af_te.fare.values, AP[0.1], AP[0.5], AP[0.9], af_base.predict(af_te[["nsmiles"]]))

airports: 180 | with coordinates: 179


train 81583 | test 8281 | test from 2025.25
airfare {"n_test": 8281, "mae_p50": 32.6379, "r2_p50": 0.5129, "mape_p50": 0.127, "coverage_p10_p90": 0.7307, "median_band_width": 80.2686, "mae_baseline": 51.473, "r2_baseline": 0.121}


## 3 · Export and parity check

Each quantile model is converted to compact JSON and re-scored with the pure-Python reference evaluator; the largest difference from XGBoost must be under one cent. Twenty "golden" rows per model are exported so the TypeScript tests can assert the same numbers.

In [9]:
def export(name, models, features, Xtest, metrics, data, extra=None):
    out = {"name": name, "version": time.strftime("%Y-%m-%d"), "features": features, "quantiles": {}, "metrics": metrics, "data": data,
           "xgboost": xgb.__version__, "trained_on": DEVICE}
    golden_X = np.asarray(Xtest[features].iloc[:20], dtype=np.float32)
    golden = {"X": golden_X.tolist()}
    worst = 0.0
    for q, m in models.items():
        cm = compact(m); key = f"p{int(q * 100)}"
        ref = m.predict(Xtest[features].iloc[:2000]); mine = predict_compact(cm, Xtest[features].iloc[:2000])
        worst = max(worst, float(np.max(np.abs(ref - mine))))
        out["quantiles"][key] = cm
        golden[key] = [round(float(v), 4) for v in m.predict(golden_X)]
    out["parity_max_abs_diff"] = worst; out["golden"] = golden
    if extra: out.update(extra)
    path = f"{OUT}/{name}.json"
    with open(path, "w") as f:
        json.dump(out, f, separators=(",", ":"))
    print(name, "parity max |diff| =", worst, "|", round(os.path.getsize(path) / 1024), "KB ->", path)
    assert worst < 0.01, "compact evaluator disagrees with XGBoost"

export("rideshare_fare", rs, RS_FEATURES, te, rs_metrics,
       {"source": "NYC TLC High Volume FHV trip records", "url": "https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page",
        "months": MONTHS, "rows_train": int(len(tr)), "target": "base_passenger_fare (USD, before tax/tolls/fees/tip)",
        "region": "New York City", "rate_card": {"intercept": float(base.intercept_), "per_mile": float(base.coef_[0]), "per_minute": float(base.coef_[1])}})

airports = st.dropna(subset=["lat", "lon"]).reset_index()
export("airfare", afm, AF_FEATURES, Xte, af_metrics,
       {"source": "U.S. DOT Consumer Airfare Report Table 1a", "url": "https://data.transportation.gov/d/tfrh-tu9e",
        "years": [int(af.year.min()), int(af.year.max())], "test_from": float(test_from), "rows_train": int(len(af_tr)),
        "target": "average fare for the airport-pair market and quarter (USD)"},
       {"airports": [{"code": r.ap, "city": r.city, "lat": round(r.lat, 4), "lon": round(r.lon, 4),
                      "log_pax": round(float(r.log_pax), 4), "fpm": round(float(r.fpm), 5)} for r in airports.itertuples()],
        "airport_defaults": med, "latest_year": int(af.year.max())})
print(sorted(os.listdir(OUT)))

rideshare_fare parity max |diff| = 0.00014173953198337585 | 1386 KB -> /net/home/chatran/smile-price-models/models/rideshare_fare.json


airfare parity max |diff| = 0.00020262200837350974 | 1936 KB -> /net/home/chatran/smile-price-models/models/airfare.json
['airfare.json', 'rideshare_fare.json']
